In [12]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer.primitives import EstimatorV2 as Estimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
import pandas as pd

n_samples = 2000  # subset — full 155K rows is impractical for QNN

# Shuffle before sampling so we get a mix of fire/no-fire rows
df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["target"].values.astype(float)

# split and scale
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

y_scaler = MinMaxScaler(feature_range=(-1, 1))
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

print(f"Train: {x_train_a.shape}, Test: {x_test_a.shape}")
print(f"y_train range: [{y_train_s.min():.3f}, {y_train_s.max():.3f}]")

Train: (1800, 9), Test: (200, 9)
y_train range: [-1.000, 1.000]


In [14]:
n_features = x_train_a.shape[1]
n_qubits = n_features

x_params = ParameterVector("x", n_features)
reps = 3  # increase to 2 or 3 once pipeline is confirmed working
theta_params = ParameterVector("θ", length=n_qubits * 2 * reps)

qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.ry(x_params[i], i)

t = 0
for r in range(reps):
    for q in range(n_qubits):
        qc.ry(theta_params[t], q); t += 1
        qc.rz(theta_params[t], q); t += 1

    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)
    if n_qubits > 2:
        qc.cx(n_qubits - 1, 0)

qc.decompose().draw("text")

global phase: (-0.5)*θ[1] - 0.5*θ[3] - 0.5*θ[5] - 0.5*θ[7] - 0.5*θ[9] - 0.5*θ[11] - 0.5*θ[13] - 0.5*θ[15] - 0.5*θ[17] - 0.5*θ[19] - 0.5*θ[21] - 0.5*θ[23] - 0.5*θ[25] - 0.5*θ[27] - 0.5*θ[29] - 0.5*θ[31] - 0.5*θ[33] - 0.5*θ[35] - 0.5*θ[37] - 0.5*θ[39] - 0.5*θ[41] - 0.5*θ[43] - 0.5*θ[45] - 0.5*θ[47] - 0.5*θ[49] - 0.5*θ[51] - 0.5*θ[53]
     ┌─────────────┐┌─────────────┐ ┌─────────┐                           »
q_0: ┤ R(x[0],π/2) ├┤ R(θ[0],π/2) ├─┤ P(θ[1]) ├───■───────────────────────»
     ├─────────────┤├─────────────┤ ├─────────┤ ┌─┴─┐     ┌──────────────┐»
q_1: ┤ R(x[1],π/2) ├┤ R(θ[2],π/2) ├─┤ P(θ[3]) ├─┤ X ├──■──┤ R(θ[20],π/2) ├»
     ├─────────────┤├─────────────┤ ├─────────┤ └───┘┌─┴─┐└──────────────┘»
q_2: ┤ R(x[2],π/2) ├┤ R(θ[4],π/2) ├─┤ P(θ[5]) ├──────┤ X ├───────■────────»
     ├─────────────┤├─────────────┤ ├─────────┤      └───┘     ┌─┴─┐      »
q_3: ┤ R(x[3],π/2) ├┤ R(θ[6],π/2) ├─┤ P(θ[7]) ├────────────────┤ X ├──────»
     ├─────────────┤├─────────────┤ ├─────────┤                └───┘      »
q_4: ┤ R(x[4],π/2) ├┤ R(θ[8],π/2) ├─┤ P(θ[9]) ├───────────────────────────»
     ├─────────────┤├─────────────┴┐├─────────┴┐                          »
q_5: ┤ R(x[5],π/2) ├┤ R(θ[10],π/2) ├┤ P(θ[11]) ├──────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                          »
q_6: ┤ R(x[6],π/2) ├┤ R(θ[12],π/2) ├┤ P(θ[13]) ├──────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                          »
q_7: ┤ R(x[7],π/2) ├┤ R(θ[14],π/2) ├┤ P(θ[15]) ├──────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                          »
q_8: ┤ R(x[8],π/2) ├┤ R(θ[16],π/2) ├┤ P(θ[17]) ├──────────────────────────»
     └─────────────┘└──────────────┘└──────────┘                          »
«                                                                     »
«q_0: ────────────────────────────────────────────────────────────────»
«       ┌──────────┐                                                  »
«q_1: ──┤ P(θ[21]) ├──────────────────────────────────────────────────»
«     ┌─┴──────────┴─┐  ┌──────────┐                                  »
«q_2: ┤ R(θ[22],π/2) ├──┤ P(θ[23]) ├──────────────────────────────────»
«     └──────────────┘┌─┴──────────┴─┐  ┌──────────┐                  »
«q_3: ───────■────────┤ R(θ[24],π/2) ├──┤ P(θ[25]) ├──────────────────»
«          ┌─┴─┐      └──────────────┘┌─┴──────────┴─┐  ┌──────────┐  »
«q_4: ─────┤ X ├─────────────■────────┤ R(θ[26],π/2) ├──┤ P(θ[27]) ├──»
«          └───┘           ┌─┴─┐      └──────────────┘┌─┴──────────┴─┐»
«q_5: ─────────────────────┤ X ├─────────────■────────┤ R(θ[28],π/2) ├»
«                          └───┘           ┌─┴─┐      └──────────────┘»
«q_6: ─────────────────────────────────────┤ X ├─────────────■────────»
«                                          └───┘           ┌─┴─┐      »
«q_7: ─────────────────────────────────────────────────────┤ X ├──────»
«                                                          └───┘      »
«q_8: ────────────────────────────────────────────────────────────────»
«                                                                     »
«                                     ┌───┐┌──────────────┐┌──────────┐     »
«q_0: ────────────────────────────────┤ X ├┤ R(θ[18],π/2) ├┤ P(θ[19]) ├──■──»
«                                     └─┬─┘└──────────────┘└──────────┘┌─┴─┐»
«q_1: ──────────────────────────────────┼──────────────────────────────┤ X ├»
«                                       │                              └───┘»
«q_2: ──────────────────────────────────┼───────────────────────────────────»
«                                       │                                   »
«q_3: ──────────────────────────────────┼───────────────────────────────────»
«                                       │                                   »
«q_4: ──────────────────────────────────┼───────────────────────────────────»
«       ┌──────────┐                    │                                 

In [15]:
observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])

estimator = Estimator()

qnn = EstimatorQNN(
    circuit=qc,
    estimator=estimator,
    observables=observable,
    input_params=list(x_params),
    weight_params=list(theta_params),
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [16]:
torch.manual_seed(0)

model = TorchConnector(qnn)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# Use a small batch for quick testing; increase for full training
"""
n_train = 10000
xtr = torch.tensor(x_train_a[:n_train], dtype=torch.float32)
ytr = torch.tensor(y_train_s[:n_train], dtype=torch.float32).view(-1, 1)
"""
xtr = torch.tensor(x_train_a, dtype=torch.float32)
ytr = torch.tensor(y_train_s, dtype=torch.float32).view(-1,1)

for epoch in range(5):
    optimizer.zero_grad()
    yhat = model(xtr)
    loss = loss_fn(yhat, ytr)
    loss.backward()
    optimizer.step()
    print(f"epoch={epoch+1:3d}, loss={loss.item():.6f}")

epoch=  1, loss=0.978086
epoch=  2, loss=0.964577
epoch=  3, loss=0.951229
epoch=  4, loss=0.934900
epoch=  5, loss=0.915528


In [19]:
Xte = torch.tensor(x_test_a, dtype=torch.float32)
with torch.no_grad():
    ypred_s = model(Xte).numpy().ravel()

# Convert predictions back to original target units
ypred = y_scaler.inverse_transform(ypred_s.reshape(-1, 1)).ravel()

# Compare with original y_test (unscaled)
# Example metrics:
from sklearn.metrics import mean_squared_error, r2_score
print("MSE:", mean_squared_error(y_test, ypred))
print("R^2:", r2_score(y_test, ypred))

MSE: 0.4384115817327769
R^2: -182.41633510195112


In [64]:
from sklearn.metrics import roc_auc_score, roc_curve

# y_true: original regression target (not scaled), shape (n,)
# y_pred: your model prediction in the same units, shape (n,)

xte = torch.tensor(x_test_a, dtype=torch.float32)

with torch.no_grad():
    y_pred_scaled = model(xte).cpu().numpy().ravel()   # shape (n_test,)
# y_scaler is the MinMaxScaler you fit on y_train earlier
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
print(y_pred)
#y_pred = 1.8-y_pred

threshold = 0.02  # example: any burned area > 0 indicates a fire
y_true_bin = (y_test > threshold).astype(int)
auc = roc_auc_score(y_true_bin, y_pred)  # y_pred can be continuous
print("AUC_ROC:", auc)

[0.6464772  0.73799795 0.5150844  0.5953821  0.53685623 0.4977748
 0.81094337 0.731794   0.78384805 0.7186847  0.55688953 0.69490135
 0.50344723 0.59969294 0.6343717  0.69974893 0.6423754  0.5935242
 0.6528446  0.55979276 0.6027459  0.5596819  0.6044681  0.7035003
 0.7262831  0.6846173  0.5681766  0.6375244  0.71560633 0.7160859
 0.63036186 0.57387346 0.62684023 0.7057542  0.6236968  0.65732026
 0.6678553  0.65892106 0.7503573  0.6102524  0.66851896 0.7952428
 0.68767506 0.59717166 0.601517   0.61449367 0.6443886  0.7084881
 0.7007484  0.6966062  0.6731228  0.7552842  0.7604931  0.6406711
 0.7356655  0.6691369  0.72486    0.59556633 0.6979202  0.7216147
 0.63704705 0.7405541  0.5982312  0.66942203 0.594929   0.6213827
 0.7473593  0.6783812  0.64957434 0.68842816 0.8097286  0.57442
 0.68606883 0.5322525  0.7018411  0.65989995 0.69076693 0.63732857
 0.67721355 0.68276054 0.52080834 0.5572622  0.77002203 0.6652412
 0.755048   0.5587836  0.68223155 0.50647515 0.5641122  0.64147586
 0.70080

In [44]:
w = model.weight.detach().cpu().numpy()   # shape: (num_weights,)
print(w.shape)
print(w)

(54,)
[-0.04839668  0.7211829  -0.5815352  -0.92589504 -0.28278005  0.3414033
 -0.240943    0.68298924  0.16046807  0.01474996 -0.55140823  0.05045009
 -0.705641   -0.7063611  -0.28476897 -0.20975378  0.5904991   0.34890443
 -0.9218238  -0.6837335   0.456243    1.0582838   0.04337547  0.952538
 -0.40578824  0.3455709   0.7997489  -1.1652304  -0.42193416 -0.49153924
 -0.41218564  0.61378914 -0.89703715 -0.22752249 -0.45028642 -0.6941028
 -0.3433253   0.85959804  0.5175206   0.48467255  0.21486963 -0.5126835
 -0.02579289 -0.9336947  -0.47607464 -0.51553     0.38223276  0.58632123
 -0.19324115 -0.03608239  0.87611204  0.9941331   0.14763744  0.13509285]
